# Fine-Tuning GPT-2 for E-Store Question Answering

In this notebook, we fine-tune a **GPT-2 model** on a custom Q&A dataset so it can answer
questions about a specific domain : an e-store in our case.

Our dataset has two columns: **question** and **answer**.

## Steps

### 1. Prepare the Dataset
We convert our CSV into a **HuggingFace Dataset** with the right format for GPT-2.
Note that each model/task expects a specific data format ;different from say a RoBERTa classifier.

### 2. Train
We use **`Trainer`** and **`TrainingArguments`** from HuggingFace to handle the training loop.

### 3. Save & Reuse
Once trained, the model is **saved to disk** for later testing or deployment.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
working_directory ="/content/drive/MyDrive/transformers/Ecommerce/ChatBot"

In [ ]:
from zipfile  import ZipFile
import os
import pandas as pd
from transformers import Trainer , TrainingArguments
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

In [ ]:
os.chdir(working_directory)

In [ ]:
os.getcwd()

'/content/drive/MyDrive/transformers/Ecommerce/ChatBot'

In [ ]:
!ls

estore_qa.zip  training_model_for_chatbot.ipynb


In [ ]:
with ZipFile("./estore_qa.zip", "r") as file:
  file.extractall()

In [ ]:
!ls

estore_qa.csv  estore_qa.zip  training_model_for_chatbot.ipynb


In [ ]:
df = pd.read_csv("./estore_qa.csv")

In [ ]:
df

,question,answer
0,What are the payment options available?,"We accept credit/debit cards, PayPal, and bank..."
1,Do you offer international shipping?,"Yes, we ship to many countries worldwide. Chec..."
2,What is the return policy?,You can return items within 30 days of receipt...
3,How can I track my order?,You can track your order using the tracking nu...
4,What should I do if I receive a damaged item?,Please contact our customer support for assist...
...,...,...
95,How do I redeem a promotional offer?,Enter the promotional code at checkout to rede...
96,What should I do if I receive an incorrect cha...,Contact support if you notice an incorrect cha...
97,Can I request a product demo?,Request a product demo by contacting our sales...
98,How do I submit a warranty claim?,Submit a warranty claim through the 'Warranty'...


# Prepare the training Dataset


In [ ]:
dataset = load_dataset("csv", data_files={"data": "./estore_qa.csv"}, split="data")

In [ ]:
dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 100
})

In [ ]:
dataset[0] # a single row looks like a dictionary where keys are the column names

{'question': 'What are the payment options available?',
 'answer': 'We accept credit/debit cards, PayPal, and bank transfers.'}

In [ ]:
model_name="gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_function(example):
  return tokenizer(example["question"], example["answer"], return_tensors="pt", padding="max_length", truncation=True,  max_length=128)


In [ ]:
tokenize_function(dataset[0]) # a dictionnary with two keys : "input_ids" and "attention_mask"

{'input_ids': tensor([[ 2061,   389,   262,  6074,  3689,  1695,    30,  1135,  2453,  3884,
            14, 11275,   270,  4116,    11, 22525,    11,   290,  3331, 16395,
            13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 5

In [ ]:
tokenized_dataset = dataset.map(tokenize_function)

In [ ]:
tokenized_dataset[0]

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["question", "answer"])

In [ ]:
tokenized_dataset[0] # each row is now tokenized and the dataset is ready for training

In [ ]:
from transformers import DataCollatorForLanguageModeling

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm = False
)

In [ ]:
working_directory

'/content/drive/MyDrive/transformers/Ecommerce/ChatBot'

In [ ]:
training_args = TrainingArguments(
    output_dir = working_directory,
    per_device_train_batch_size= 6,
    num_train_epochs= 15,
    learning_rate= 5e-5,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model= model,
    train_dataset= tokenized_dataset,
    args= training_args,
    data_collator = data_collator
)

In [ ]:
trainer.train() # train the model

In [ ]:
model_path = working_directory + "/model_dir"

trainer.save_model(model_path)

tokenizer.save_pretrained(model_path)